In [3]:
import sys
sys.path.insert(0, "../..")
sys.modules['__main__'].__file__ = "/home/user/Documents/plyt.py"

if sys.version_info[0] >= 3:
    raw_input = input

import ply.lex as lex
import ply.yacc as yacc
import os


class Parser(object):
    """
    Base class for a lexer/parser that has the rules defined as methods
    """
    tokens = ()
    precedence = ()

    def __init__(self, **kw):
        self.debug = kw.get('debug', 0)
        self.names = {}
        try:
            modname = os.path.split(os.path.splitext(__file__)[0])[
                1] + "_" + self.__class__.__name__
        except:
            modname = "parser" + "_" + self.__class__.__name__
        self.debugfile = modname + ".dbg"
        self.tabmodule = modname + "_" + "parsetab"
        # print self.debugfile, self.tabmodule

        # Build the lexer and parser
        lex.lex(module=self, debug=self.debug)
        yacc.yacc(module=self,
                  debug=self.debug,
                  debugfile=self.debugfile,
                  tabmodule=self.tabmodule)

    def run(self, inputg = None):
        if inputg is not None:
            yacc.parse(inputg)
            return
        while 1:
            try:
                s = raw_input('calc > ')
            except EOFError:
                break
            if not s:
                continue
            yacc.parse(s)


class Calc(Parser):
        
    tokens = [
        'NAME', 'NUMBER', 'SEP',
        'LPAREN', 'RPAREN', 'COMMA'
    ]
    
    RESERVED = {
        "stats": "STATS",
        "min": "MIN",
        "max": "MAX",
        "avg": "AVG",
        "std": "STD",
        "by": "BY",
        "as": "AS",
        "table" : "TABLE", 
        "where": "WHERE",
        "index": "INDEX",
        "if": "IF",
        "case": "CASE",
        "eval": "EVAL",
    }
    
    tokens += RESERVED.values()

    # Tokens
    t_SEP = r'\|'
    t_LPAREN = r'\('
    t_RPAREN = r'\)'
    t_COMMA = r','
    t_NAME = r'[a-zA-Z_][a-zA-Z0-9_]*'

    def t_NUMBER(self, t):
        r'\d+'
        try:
            t.value = int(t.value)
        except ValueError:
            print("Integer value too large %s" % t.value)
            t.value = 0
        # print "parsed number %s" % repr(t.value)
        return t

    def t_NAME(self, t):
        r'[a-zA-Z_][a-zA-Z_0-9]*'
        t.type = Calc.RESERVED.get(t.value, 'NAME')    # Check for reserved words
        return t
    
    t_ignore = " \t"

    def t_newline(self, t):
        r'\n+'
        t.lexer.lineno += t.value.count("\n")

    def t_error(self, t):
        print("Illegal character '%s'" % t.value[0])
        t.lexer.skip(1)

    start = "statement"
    
    def p_statement(self, p):
        """statement : INDEX SEP commands
                     | INDEX """
        if len(p)<3:
            ret = "df"
        else:
            ret = "df.%s"%(p[3])
        print(ret)
        
    def p_commands(self, p):
        """commands : commands SEP commandst
                   | commandst """
        if len(p)<3:
            p[0] = p[1]
        else:
            p[0] = "%s.%s"%(p[1], p[3])      
        
    def p_statement_table(self, p):
        """ commandst : TABLE argfields """
        p[0] = "select(%s)"%p[2]
        
    def p_statement_eval(self, p):
        """commandst : EVAL evalst"""
        p[0] = "withColumn(%s)"%p[2]
        
    def p_statement_evalst(self, p):
        """evalst : argfields"""
        # fix this
        p[0] = "%s"%p[1]
        
    def p_statement_where(self, p):
        """commandst : WHERE conditionalst"""
        p[0] = "filter(%s)"%p[2]
        
    def p_statement_condition(self, p):
        """conditionalst : argfields"""
        # fix this
        p[0] = "%s"%p[1]
        
    def p_statement_stats(self, p):
        """commandst : STATS aggstats BY argfields"""
        #df.groupBy("col1").agg(sum("col2").alias("col2")
        p[0] = "groupBy(%s).agg(%s)"%(p[4], p[2])
        
    def p_statement_agg(self, p):
        """aggstats : aggstats COMMA aggfuncexpr 
                    | aggfuncexpr"""
        if len(p)<3:
            p[0] = p[1]
        else:
            p[0] = "%s,%s"%(p[1], p[3])

    def p_statement_aggfuncexpr(self, p):
        """aggfuncexpr : aggfunc LPAREN expression RPAREN AS expression
                        | aggfunc LPAREN expression RPAREN """
        if len(p)<6:
            p[0] = "%s(%s)"%(p[1], p[3])
        else:
            p[0] = "%s(%s).alias(%s)"%(p[1], p[3], p[6])
        
    def p_statement_aggfunc(self, p):
        """aggfunc : MIN
                    | MAX
                    | AVG
                    | STD """
        p[0] = p[1]
        
    def p_statement_argstats(self, p):
        """argfields : argfields COMMA NAME
                    | NAME """
        p[0] = ",".join([p[i] for i in range(1, len(p), 2)])
        
    def p_expression_name(self, p):
        'expression : NAME'
        try:
            p[0] = p[1]
        except LookupError:
            print("Undefined name '%s'" % p[1])
            p[0] = 0

    def p_error(self, p):
        if p:
            print("Syntax error at '%s'" % p.value)
        else:
            print("Syntax error at EOF")

if __name__ == '__main__':
    calc = Calc()
    #calc.run("stats min(f1) as mf1 max(f2) as mf2, values(f3) as f3s by f4, f5")
    calc.run("""index 
    | table a1, a2
    | stats max(a1) as a11, avg(a2) as a21 by a5, a6 
    | eval a5""")
    #calc.run("index | table a1, a2, a3, a4, a5, a6")
    #calc.run("stats max(a1) as a11, min(a2) as a21 by a5, a6")

df.select(a1,a2).groupBy(a5,a6).agg(max(a1).alias(a11),avg(a2).alias(a21)).withColumn(a5)
